In [1]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv

!mpirun --version | head -n 2

!pip install -q mpi4py numba numpy

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-2ee934ca-5301-564d-a59b-c9bf5893bf73, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6



In [3]:
GITHUB_USERNAME = "asemabiseva"

assert (
    GITHUB_USERNAME != "YOUR_GITHUB_USERNAME"
    and len(GITHUB_USERNAME.strip()) > 0
), "Enter your actual GitHub username!"

print("Configured GitHub identity:", GITHUB_USERNAME)

Configured GitHub identity: asemabiseva


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
%%writefile /content/drive/MyDrive/distributed_cuda_solver.py
import sys
import os
import json
import math
import hashlib
import subprocess
import time
import re
import numpy as np
from mpi4py import MPI
from numba import cuda, float64

GLOBAL_NX = 512
GLOBAL_NY = 512
LOCAL_NX = GLOBAL_NX // 2
LOCAL_NY = GLOBAL_NY // 2
PAD_NX = LOCAL_NX + 2
PAD_NY = LOCAL_NY + 2
TILE = 16
SHARED = 18
STEPS = 400
LAMBDA_DAMP = 0.15
DX = 1.0 / 513.0
DY = DX
KAPPA_MIN = 0.08
KAPPA_MAX = 0.24
DT = 0.20 * DX * DX / (4.0 * KAPPA_MAX)
TPB_1D = 256


def get_gpu_uuid():
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        encoding="utf-8"
    )
    return out.strip().splitlines()[0].strip()


def derive_student_seed(github_user, gpu_uuid):
    digest = hashlib.sha256(
        f"{github_user.strip()}::{gpu_uuid}".encode("utf-8")
    ).hexdigest()
    return int(digest[:16], 16)


def init_distributed_context(github_user):
    comm = MPI.COMM_WORLD
    rank, size = comm.Get_rank(), comm.Get_size()
    if size != 4:
        raise RuntimeError(f"Expected exactly 4 MPI ranks, got {size}")
    cuda.select_device(0)
    gpu_uuid = get_gpu_uuid() if rank == 0 else None
    gpu_uuid = comm.bcast(gpu_uuid, root=0)
    seed = derive_student_seed(github_user, gpu_uuid)

    cart = comm.Create_cart(
        dims=[2, 2], periods=[False, False], reorder=False
    )
    coords = cart.Get_coords(cart.Get_rank())
    up, down = cart.Shift(0, 1)
    left, right = cart.Shift(1, 1)
    if rank == 0:
        print(f"MPI topology: {size} ranks in 2x2 Cartesian grid")
        print(f"GPU UUID: {gpu_uuid}")
        print(f"INSTANCE_SEED: {seed}")
    return comm, cart, coords, (up, down, left, right), seed, gpu_uuid


def initialize_subdomain(coords, seed, neighbors):
    row_rank, col_rank = coords
    # Padded local coordinates correspond to global grid indices 0..513.
    gy = row_rank * LOCAL_NY + np.arange(PAD_NY, dtype=np.float64)
    gx = col_rank * LOCAL_NX + np.arange(PAD_NX, dtype=np.float64)
    x = gx[None, :] / 513.0
    y = gy[:, None] / 513.0
    rng = np.random.default_rng(seed + row_rank * 17 + col_rank * 101)

    # Smooth heterogeneous diffusivity in the required interval.
    kappa = 0.16 + 0.035 * np.sin(2.0 * np.pi * x) * np.cos(2.0 * np.pi * y)
    kappa += 0.025 * np.sin(4.0 * np.pi * y + 0.3)
    kappa = np.clip(kappa, KAPPA_MIN, KAPPA_MAX).astype(np.float64)

    # Seed-derived Gaussian pulse locations and amplitudes.
    centers = rng.uniform(0.15, 0.85, size=(3, 2))
    amplitudes = rng.uniform(0.5, 1.0, size=3)
    u = np.zeros((PAD_NY, PAD_NX), dtype=np.float64)
    for (cx, cy), amp in zip(centers, amplitudes):
        u += amp * np.exp(-((x - cx) ** 2 + (y - cy) ** 2) / 0.008)
    source = np.zeros_like(u)
    sx, sy = 0.5, 0.5
    source += 0.08 * np.exp(-((x - sx) ** 2 + (y - sy) ** 2) / 0.002)

    # Global physical Dirichlet boundaries are zero.
    up, down, left, right = neighbors
    if up == MPI.PROC_NULL:
        u[0, :] = 0.0
    if down == MPI.PROC_NULL:
        u[-1, :] = 0.0
    if left == MPI.PROC_NULL:
        u[:, 0] = 0.0
    if right == MPI.PROC_NULL:
        u[:, -1] = 0.0
    return u, kappa, source


@cuda.jit
def pack_halos_kernel(u, send_up, send_down, send_left, send_right, ny, nx):
    i = cuda.grid(1)
    if i < nx:
        send_up[i] = u[1, i + 1]
        send_down[i] = u[ny, i + 1]
    if i < ny:
        send_left[i] = u[i + 1, 1]
        send_right[i] = u[i + 1, nx]


@cuda.jit
def unpack_halos_kernel(u, recv_up, recv_down, recv_left, recv_right,
                        ny, nx, has_up, has_down, has_left, has_right):
    i = cuda.grid(1)
    if i < nx:
        if has_up:
            u[0, i + 1] = recv_up[i]
        if has_down:
            u[ny + 1, i + 1] = recv_down[i]
    if i < ny:
        if has_left:
            u[i + 1, 0] = recv_left[i]
        if has_right:
            u[i + 1, nx + 1] = recv_right[i]


@cuda.jit
def tiled_stencil_kernel(u, kappa, source, out, ny, nx, dt, dx, dy,
                         r_start, r_end, c_start, c_end):
    # Each block covers a 16x16 output tile and loads an 18x18 u tile.
    smem = cuda.shared.array(shape=(18, 18), dtype=float64)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    row0 = r_start + cuda.blockIdx.y * 16
    col0 = c_start + cuda.blockIdx.x * 16

    # Cooperatively load all 324 shared-memory positions.
    tid = ty * 16 + tx
    for q in range(tid, 18 * 18, 16 * 16):
        sy = q // 18
        sx = q % 18
        gr = row0 + sy - 1
        gc = col0 + sx - 1
        if gr < 0:
            gr = 0
        if gr > ny + 1:
            gr = ny + 1
        if gc < 0:
            gc = 0
        if gc > nx + 1:
            gc = nx + 1
        smem[sy, sx] = u[gr, gc]

    cuda.syncthreads()

    r = row0 + ty
    c = col0 + tx
    if r >= r_start and r < r_end and c >= c_start and c < c_end:
        center = smem[ty + 1, tx + 1]
        north = smem[ty, tx + 1]
        south = smem[ty + 2, tx + 1]
        west = smem[ty + 1, tx]
        east = smem[ty + 1, tx + 2]

        kc = kappa[r, c]
        kn = 0.5 * (kc + kappa[r - 1, c])
        ks = 0.5 * (kc + kappa[r + 1, c])
        kw = 0.5 * (kc + kappa[r, c - 1])
        ke = 0.5 * (kc + kappa[r, c + 1])

        flux_x = (ke * (east - center) - kw * (center - west)) / (dx * dx)
        flux_y = (ks * (south - center) - kn * (center - north)) / (dy * dy)
        out[r, c] = center + dt * (
            flux_x + flux_y - 0.15 * center * center * center + source[r, c]
        )


def make_device_state(u_host, k_host, s_host):
    return (
        cuda.to_device(u_host),
        cuda.to_device(k_host),
        cuda.to_device(s_host),
        cuda.device_array_like(u_host)
    )


def exchange_halos_async(cart, neighbors, u_dev, streams, pinned, dsend, drecv):
    up, down, left, right = neighbors
    stream_boundary = streams["boundary"]
    ny, nx = LOCAL_NY, LOCAL_NX
    pack_halos_kernel[(1,), (TPB_1D,), stream_boundary](
        u_dev, dsend[0], dsend[1], dsend[2], dsend[3], ny, nx
    )
    for j in range(4):
        dsend[j].copy_to_host(pinned["send"][j], stream=stream_boundary)
    stream_boundary.synchronize()

    # Receive from each neighbor into the corresponding ghost-side buffer.
    recv_specs = [
        (up, 0, 11),       # from up: neighbor's down-going message
        (down, 1, 10),     # from down: neighbor's up-going message
        (left, 2, 13),     # from left: neighbor's right-going message
        (right, 3, 12),    # from right: neighbor's left-going message
    ]
    send_specs = [
        (up, 0, 10), (down, 1, 11), (left, 2, 12), (right, 3, 13)
    ]
    requests = []
    for neighbor, buf_index, tag in recv_specs:
        if neighbor != MPI.PROC_NULL:
            requests.append(cart.Irecv(pinned["recv"][buf_index], source=neighbor, tag=tag))
        else:
            pinned["recv"][buf_index].fill(0.0)
    for neighbor, buf_index, tag in send_specs:
        if neighbor != MPI.PROC_NULL:
            requests.append(cart.Isend(pinned["send"][buf_index], dest=neighbor, tag=tag))
    return requests


def allocate_exchange_buffers():
    send_h = [cuda.pinned_array(LOCAL_NX, dtype=np.float64) for _ in range(4)]
    recv_h = [cuda.pinned_array(LOCAL_NX, dtype=np.float64) for _ in range(4)]
    send_d = [cuda.device_array(LOCAL_NX, dtype=np.float64) for _ in range(4)]
    recv_d = [cuda.device_array(LOCAL_NX, dtype=np.float64) for _ in range(4)]
    # Row buffers have nx elements; column buffers have ny (equal here).
    return {"send": send_h, "recv": recv_h}, send_d, recv_d


def post_received_halos(u_dev, pinned, drecv, stream):
    for j in range(4):
        drecv[j].copy_to_device(pinned["recv"][j], stream=stream)
    unpack_halos_kernel[(1,), (TPB_1D,), stream](
        u_dev, drecv[0], drecv[1], drecv[2], drecv[3],
        LOCAL_NY, LOCAL_NX,
        True, True, True, True
    )


def launch_box(u, k, s, out, stream, box):
    r0, r1, c0, c1 = box
    blocks = (
        math.ceil((c1 - c0) / TILE),
        math.ceil((r1 - r0) / TILE)
    )
    tiled_stencil_kernel[blocks, (TILE, TILE), stream](
        u, k, s, out, LOCAL_NY, LOCAL_NX, DT, DX, DY, r0, r1, c0, c1
    )


def compute_full_stencil(u, k, s, out, stream):
    launch_box(u, k, s, out, stream, (1, LOCAL_NY + 1, 1, LOCAL_NX + 1))


def compute_overlap_step(cart, neighbors, state, streams, pinned, dsend, drecv):
    u, k, s, out = state
    sb = streams["boundary"]
    si = streams["interior"]

    requests = exchange_halos_async(cart, neighbors, u, streams, pinned, dsend, drecv)

    # Deep interior does not depend on newly received ghost cells.
    launch_box(u, k, s, out, si, (2, LOCAL_NY, 2, LOCAL_NX))

    MPI.Request.Waitall(requests)
    post_received_halos(u, pinned, drecv, sb)

    # Four perimeter strips; corners are included once by top/bottom strips.
    launch_box(u, k, s, out, sb, (1, 2, 1, LOCAL_NX + 1))
    launch_box(u, k, s, out, sb, (LOCAL_NY, LOCAL_NY + 1, 1, LOCAL_NX + 1))
    launch_box(u, k, s, out, sb, (2, LOCAL_NY, 1, 2))
    launch_box(u, k, s, out, sb, (2, LOCAL_NY, LOCAL_NX, LOCAL_NX + 1))

    si.synchronize()
    sb.synchronize()
    return out, u


def compute_serial_step(cart, neighbors, state, streams, pinned, dsend, drecv):
    u, k, s, out = state
    sb = streams["boundary"]
    requests = exchange_halos_async(cart, neighbors, u, streams, pinned, dsend, drecv)
    MPI.Request.Waitall(requests)
    post_received_halos(u, pinned, drecv, sb)
    compute_full_stencil(u, k, s, out, sb)
    sb.synchronize()
    return out, u


def global_diagnostics(cart, u_dev, step):
    u = u_dev.copy_to_host()
    local_energy = float(np.sum(u[1:-1, 1:-1]) * DX * DY)
    local_sq = float(np.sum(u[1:-1, 1:-1] ** 2))
    energy = cart.allreduce(local_energy, op=MPI.SUM)
    norm = math.sqrt(cart.allreduce(local_sq, op=MPI.SUM))
    if cart.Get_rank() == 0:
        print(f"step={step:3d} energy={energy:.16e} L2={norm:.16e}")
    return energy, norm


def run_solver(cart, coords, neighbors, seed, mode):
    u0, k_host, s_host = initialize_subdomain(coords, seed, neighbors)
    state = make_device_state(u0, k_host, s_host)
    streams = {"boundary": cuda.stream(), "interior": cuda.stream()}
    pinned, dsend, drecv = allocate_exchange_buffers()
    energies = [(0, *global_diagnostics(cart, state[0], 0))]

    # CUDA events capture elapsed GPU work for this rank.
    start_event = cuda.event(timing=True)
    end_event = cuda.event(timing=True)
    t0 = MPI.Wtime()
    for step in range(1, STEPS + 1):
        start_event.record(streams["boundary"])
        if mode == "serialized":
            out, old = compute_serial_step(
                cart, neighbors, state, streams, pinned, dsend, drecv
            )
        else:
            out, old = compute_overlap_step(
                cart, neighbors, state, streams, pinned, dsend, drecv
            )
        end_event.record(streams["boundary"])
        end_event.synchronize()
        state = (out, state[1], state[2], old)
        if step % 100 == 0:
            energies.append((step, *global_diagnostics(cart, state[0], step)))
    elapsed = MPI.Wtime() - t0
    gpu_event_ms = cuda.event_elapsed_time(start_event, end_event)
    return state[0], elapsed, gpu_event_ms, energies, float(np.max(k_host))


def ptx_audit():
    # Force compilation before inspecting the generated PTX.
    dummy = cuda.to_device(np.zeros((PAD_NY, PAD_NX), dtype=np.float64))
    kd = cuda.to_device(np.full((PAD_NY, PAD_NX), 0.16, dtype=np.float64))
    sd = cuda.to_device(np.zeros((PAD_NY, PAD_NX), dtype=np.float64))
    od = cuda.device_array_like(dummy)
    tiled_stencil_kernel[(1, 1), (16, 16)](
        dummy, kd, sd, od, LOCAL_NY, LOCAL_NX, DT, DX, DY, 1, 2, 1, 2
    )
    cuda.synchronize()
    asm_obj = tiled_stencil_kernel.inspect_asm()
    asm = "\n".join(asm_obj.values()) if isinstance(asm_obj, dict) else str(asm_obj)
    counts = {
        "st_shared_f64": len(re.findall(r"st\.shared\.f64", asm)),
        "ld_shared_f64": len(re.findall(r"ld\.shared\.f64", asm)),
        "bar_sync": len(re.findall(r"bar\.sync", asm)),
        "st_shared_any": len(re.findall(r"st\.shared", asm)),
        "ld_shared_any": len(re.findall(r"ld\.shared", asm)),
    }
    return counts


def main():
    if len(sys.argv) < 2 or not sys.argv[1].strip():
        raise RuntimeError("Pass your exact GitHub username as the first argument.")
    github_user = sys.argv[1].strip()
    comm, cart, coords, neighbors, seed, gpu_uuid = init_distributed_context(github_user)

    if cart.Get_rank() == 0:
        print(f"Local padded array: {PAD_NY}x{PAD_NX}; dt={DT:.8e}; dx={DX:.8e}")
    serialized, serial_s, serial_gpu_ms, serial_energy, kmax1 = run_solver(
        cart, coords, neighbors, seed, "serialized"
    )
    overlapped, overlap_s, overlap_gpu_ms, overlap_energy, kmax2 = run_solver(
        cart, coords, neighbors, seed, "overlap"
    )

    local_diff = float(np.max(np.abs(
        serialized.copy_to_host() - overlapped.copy_to_host()
    )))
    parity = cart.allreduce(local_diff, op=MPI.MAX)
    max_k = cart.allreduce(max(kmax1, kmax2), op=MPI.MAX)
    serial_s_max = cart.allreduce(serial_s, op=MPI.MAX)
    overlap_s_max = cart.allreduce(overlap_s, op=MPI.MAX)
    serial_gpu_max = cart.allreduce(serial_gpu_ms, op=MPI.MAX)
    overlap_gpu_max = cart.allreduce(overlap_gpu_ms, op=MPI.MAX)

    audit = ptx_audit()
    audit_all = cart.gather(audit, root=0)  # Metadata only; no field arrays gathered.
    if cart.Get_rank() == 0:
        ptx = audit_all[0]
        manifest = {
            "github_username": github_user,
            "gpu_uuid": gpu_uuid,
            "instance_seed": int(seed),
            "mpi_ranks": 4,
            "cartesian_dims": [2, 2],
            "local_padded_shape": [PAD_NY, PAD_NX],
            "steps": STEPS,
            "dt": DT,
            "dx": DX,
            "max_diffusivity": max_k,
            "serialized_ms": serial_s_max * 1000.0,
            "overlapped_ms": overlap_s_max * 1000.0,
            "serialized_cuda_event_ms_last_step": serial_gpu_max,
            "overlapped_cuda_event_ms_last_step": overlap_gpu_max,
            "parity_max_diff": parity,
            "ptx_counts": ptx,
            "ptx_audit_passed": (
                ptx["st_shared_any"] > 0 and
                ptx["ld_shared_any"] > 0 and
                ptx["bar_sync"] > 0
            ),
            "energy_trace_serialized": [
                {"step": int(v[0]), "energy": float(v[1]), "l2": float(v[2])}
                for v in serial_energy
            ],
            "energy_trace_overlapped": [
                {"step": int(v[0]), "energy": float(v[1]), "l2": float(v[2])}
                for v in overlap_energy
            ],
        }
        with open("submission_manifest.json", "w", encoding="utf-8") as f:
            json.dump(manifest, f, indent=2)
        print("\n=== FINAL RESULTS ===")
        print(json.dumps(manifest, indent=2))
        if parity >= 1e-12:
            raise RuntimeError(f"Serialized/overlapped parity failed: {parity}")
        if not manifest["ptx_audit_passed"]:
            raise RuntimeError("PTX shared-memory/synchronization audit failed.")
        print("SUCCESS: submission_manifest.json generated.")
    cart.Barrier()
    cart.Free()
    if MPI.Is_initialized() and not MPI.Is_finalized():
        MPI.Finalize()


if __name__ == "__main__":
    main()


Overwriting /content/drive/MyDrive/distributed_cuda_solver.py


In [18]:
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "/content/drive/MyDrive/distributed_cuda_solver.py",
    GITHUB_USERNAME
]

proc = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(proc.stdout)

if proc.returncode != 0:
    print("ERROR:")
    print(proc.stderr)

    raise RuntimeError(
        f"MPI execution failed: {proc.returncode}"
    )

MPI topology: 4 ranks in 2x2 Cartesian grid
GPU UUID: GPU-2ee934ca-5301-564d-a59b-c9bf5893bf73
INSTANCE_SEED: 16454080969576568231
Local padded array: 258x258; dt=7.91633260e-07; dx=1.94931774e-03
step=  0 energy=5.2697421631496058e-02 L2=9.2420274188117503e+01
step=100 energy=5.2693859336377380e-02 L2=9.1170905512238022e+01
step=200 energy=5.2691551730880901e-02 L2=9.0475191149162313e+01
step=300 energy=5.2689544493703160e-02 L2=8.9879164561843140e+01
step=400 energy=5.2687686874263243e-02 L2=8.9337007165269341e+01
step=  0 energy=5.2697421631496058e-02 L2=9.2420274188117503e+01
step=100 energy=5.2693859336377380e-02 L2=9.1170905512238022e+01
step=200 energy=5.2691551730880901e-02 L2=9.0475191149162313e+01
step=300 energy=5.2689544493703160e-02 L2=8.9879164561843140e+01
step=400 energy=5.2687686874263243e-02 L2=8.9337007165269341e+01

=== FINAL RESULTS ===
{
  "github_username": "asemabiseva",
  "gpu_uuid": "GPU-2ee934ca-5301-564d-a59b-c9bf5893bf73",
  "instance_seed": 164540809695765

##Question 1. Memory Coalescing

The array is stored row by row, so elements in the same row are next to each other. However, column elements are separated by 258 elements (2064 bytes). Because of this, they cannot be copied as one continuous block. pack_halos_kernel collects the column values into continuous buffers before transferring them. One column contains 2048 useful bytes, but copying the full bounding box transfers 528384 bytes. This means 526336 extra bytes per column, or 1052672 bytes for both columns.

##Question 2. Shared Memory

Kernel uses an 18×18 shared-memory tile. Thread (0,0) writes smem[0,0] and smem[14,4], while thread (15,15) writes smem[14,3]. They read the center and four neighboring values from shared memory. On the Tesla T4, there are 32 banks, each 4 bytes wide. Since float64 uses 8 bytes, it occupies two banks. This can cause 2-way bank conflicts and make memory access slower.

##Question 3. PTX Audit

My PTX audit showed 5 st.shared.f64, 5 ld.shared.f64 and 1 bar.sync instructions. The audit passed. If cuda.syncthreads() is removed, some warps may read shared memory before other warps finish writing it. This can lead to incorrect results because some values are not ready yet.

##Question 4. GPU Performance

Four MPI processes shared one Tesla T4 GPU. Each process has its own CUDA context, but they share the GPU resources. My serialized execution took 4635.37 ms, while the overlapped version took 2105.70 ms. The overlapped version was about 2.2 times faster because calculations could happen while communication was running. However, all processes still shared the same GPU, unlike a system with four separate GPUs.

##Question 5. MPI Deadlock

If all ranks use blocking Send before Recv, deadlock can happen with large messages. MPI may wait for the receiver before completing the send. If every rank is waiting to send, nobody reaches the receive operation. MPI.PROC_NULL makes boundary communication easier. For example, Rank (0,0) has no upper or left neighbor, so MPI simply ignores communication with these missing neighbors.

##Question 6. CFL Stability

My maximum diffusivity was 0.2148497945, and dx was 1/513 = 0.0019493177. The time step used was 7.91633 × 10⁻⁷. Using the formula dt_max = dx² / (4 × κmax), the maximum stable time step is approximately 4.42151 × 10⁻⁶. My time step is smaller, so it satisfies the stability condition. The energy also decreased from 0.05269742 at step 0 to 0.05268769 at step 400. This shows the expected dissipative behavior of the diffusion process. The serialized and overlapped results were identical.